# FloorPlan ML — Phase 1: SegFormer Training

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** (free) or **A100** (Pro)
2. Connect to Google Drive (optional, for checkpoint persistence)

**Total time:**
- T4 (free):  ~12 hours for 50 epochs on CubiCasa5k
- A100 (Pro): ~2 hours


In [ ]:
# ── Step 1: Check GPU ─────────────────────────────────────────────────────
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ── Step 2: Mount Google Drive (for checkpoint persistence) ───────────────
# Skip this if you don't want to save checkpoints to Drive
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/floorplan_ml/checkpoints'

In [ ]:
# ── Step 3: Install dependencies ──────────────────────────────────────────
!pip install -q transformers albumentations huggingface_hub tensorboard svgpathtools lxml

In [ ]:
# ── Step 4: Clone your floorplan-ml repo ──────────────────────────────────
# Option A: Clone from GitHub
# !git clone https://github.com/YOUR_USERNAME/floorplan-ml.git
# %cd floorplan-ml

# Option B: Upload src/ files directly to Colab
# Use the file upload widget on the left sidebar
import sys
sys.path.insert(0, '/content/floorplan-ml/src')

In [ ]:
# ── Step 5: Download CubiCasa5k ───────────────────────────────────────────
!git clone https://github.com/CubiCasa/CubiCasa5k.git /content/floorplan-ml/data/raw/cubicasa5k
import sys
sys.path.insert(0, '/content/floorplan-ml/data/raw/cubicasa5k')

In [ ]:
# ── Step 6: Prepare data (SVG → masks) ────────────────────────────────────
%cd /content/floorplan-ml
!python tools/prepare_cubicasa.py
# Takes ~10-15 min on Colab

In [ ]:
# ── Step 7: Verify data ───────────────────────────────────────────────────
import json
with open('data/processed/splits/train.json') as f:
    train = json.load(f)
print(f'Train samples: {len(train)}')
print(f'First sample: {train[0]}')

In [ ]:
# ── Step 8: Preview augmentations ─────────────────────────────────────────
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from augmentations import get_train_transforms

sample = train[0]
img  = np.array(Image.open(sample['image_path']).convert('RGB'))
mask = np.array(Image.open(sample['mask_path']).convert('L'))
tf   = get_train_transforms(512)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    aug = tf(image=img, mask=mask)
    disp = aug['image'] * np.array([0.229,0.224,0.225]) + np.array([0.485,0.456,0.406])
    disp = np.clip(disp, 0, 1)
    axes[0,i].imshow(disp); axes[0,i].axis('off'); axes[0,i].set_title(f'Image {i+1}')
    cmap = {0:[240,240,240], 1:[30,30,30], 2:[50,100,200], 3:[50,180,80]}
    mrgb = np.array([[cmap[v] for v in row] for row in aug['mask']], dtype=np.uint8)
    axes[1,i].imshow(mrgb); axes[1,i].axis('off'); axes[1,i].set_title(f'Mask {i+1}')
plt.tight_layout(); plt.show()

In [ ]:
# ── Step 9: Train SegFormer ────────────────────────────────────────────────
# Adjust batch_size if CUDA OOM: T4=4, A100=16
import subprocess
result = subprocess.run([
    'python', 'src/train_segformer.py',
    '--epochs', '50',
    '--batch-size', '8',
    '--save-dir', CHECKPOINT_DIR + '/segformer',
], capture_output=False)
# Training output will stream to this cell

In [ ]:
# ── Step 10: Test on a floor plan image ───────────────────────────────────
# Upload a test floor plan first (Files panel on left)
from ml_preprocessor import MLPreprocessor
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

TEST_IMAGE = '/content/test_floorplan.png'  # change this path
CHECKPOINT = CHECKPOINT_DIR + '/segformer/best'

pre   = MLPreprocessor(CHECKPOINT)
masks = pre.get_masks(TEST_IMAGE)
clean = pre.make_clean_wall_image(TEST_IMAGE)
stats = pre.get_confidence_stats(TEST_IMAGE)

print(f"Wall: {stats['wall_ratio']*100:.1f}%  Door: {stats['door_ratio']*100:.1f}%  Win: {stats['window_ratio']*100:.1f}%")
print(f"Valid: {stats['looks_valid']}  Confidence: {stats['mean_confidence']:.3f}")

orig = np.array(Image.open(TEST_IMAGE).convert('RGB'))
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(orig); axes[0].set_title('Original'); axes[0].axis('off')
pred_rgb = np.zeros((*masks['full_pred'].shape, 3), dtype=np.uint8)
pred_rgb[masks['full_pred']==1] = [30,30,30]
pred_rgb[masks['full_pred']==2] = [50,100,200]
pred_rgb[masks['full_pred']==3] = [50,180,80]
pred_rgb[masks['full_pred']==0] = [240,240,240]
axes[1].imshow(pred_rgb); axes[1].set_title('Prediction'); axes[1].axis('off')
import cv2
axes[2].imshow(cv2.cvtColor(clean, cv2.COLOR_BGR2RGB)); axes[2].set_title('Clean (raster_parser input)'); axes[2].axis('off')
plt.tight_layout(); plt.show()